# U-Net на кропах 256 з повної роздільності

Етап 9, покращення 1. Копія `05_unet_train.ipynb`, змінено лише **дані**:

| | v1 (`05`) | v2 (цей ноутбук) |
|---|---|---|
| train | цілий знімок, стиснутий 768 → 384 | випадковий кроп 256×256 з оригіналу 768 (`ShipCropDataset`) |
| val | 384 | **768 цілком** — так само, як буде на test |
| масштаб кораблів | у 2 рази менші за оригінал | оригінальний |

Модель, loss, оптимізатор, LR, старт з ImageNet — ті самі, що у v1: змінюємо одну річ за раз.

**Чому має допомогти:** при стисканні до 384 корабель шириною 4 пікселі стає 2-піксельним, а оракул на
етапі 7 показав, що сама роздільність 384 обмежує F2 (F2-score — F-міра з вагою recall вдвічі більшою за precision)
до 0.743 навіть з ідеальною маскою. Кроп зберігає всі пікселі, а розмір входу — менший, ніж 384.

**Val Dice v2 і v1 напряму не порівнюються:** v1 міряв Dice на масках 384, v2 — на 768 (дрібні деталі,
які v1 «не бачив», тепер враховуються). Чесне порівняння — F2 пайплайну (крок 3).

**На Kaggle:**
1. Settings → Accelerator → **GPU**, Internet → **On**; секрет `KAGGLE_API_TOKEN` прикріплений;
   Input — `airbus-ship-detection-src` **останньої версії** (з `ShipCropDataset` у `dataset.py`).
2. Пробний запуск: `PILOT = True` → **Run All** в інтерактивній сесії → дивимось оцінку часу епохи.
3. Повне навчання: `PILOT = False`, `EPOCHS` — за результатом пілота → **Save Version → Save & Run All**.

## 0. Дані на Kaggle

Як у `05`.

In [ ]:
import os
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
KAGGLE_DATA_DIR = Path("/kaggle/tmp/data")

if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    os.environ["KAGGLE_API_TOKEN"] = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
    # Встановлена в Kaggle версія kaggle може не знати нового формату токена.
    !pip install -q kaggle==2.2.4 segmentation-models-pytorch==0.5.0

In [ ]:
if ON_KAGGLE and not (KAGGLE_DATA_DIR / "train_v2").exists():
    !mkdir -p /kaggle/tmp && cd /kaggle/tmp && kaggle competitions download -c airbus-ship-detection
    !unzip -q /kaggle/tmp/airbus-ship-detection.zip "train_v2/*" train_ship_segmentations_v2.csv -d {KAGGLE_DATA_DIR}

if ON_KAGGLE:
    n_files = sum(1 for _ in (KAGGLE_DATA_DIR / "train_v2").iterdir())
    print("знімків у train_v2:", n_files)
    assert n_files == 192556

## 1. Модель і цикл навчання

| | Локально (CPU) | Kaggle, `PILOT = True` | Kaggle, `PILOT = False` |
|---|---|---|---|
| знімки | `data/train_micro/`, лише з кораблями | `/kaggle/tmp/data/train_v2/` | те саме |
| train | ~120 кропів, 2 батчі | 50 батчів по 16 кропів | ~38 300 кропів (1 на знімок) |
| val | ~30 знімків 768 | 1 000 випадкових з кораблями | усі 4 256 з кораблями |
| епох | 1 | 1 | `EPOCHS` |
| результати | `data/unet_crops_local/` | `/kaggle/working/` | `/kaggle/working/` |

Батч val — 8, а не 16: знімок 768² має в 4 рази більше пікселів, ніж 384², і активації decoder
займають у стільки ж разів більше пам'яті GPU.

`EPOCHS = 12` — попередньо: кроп 256² має в 2.25 раза менше пікселів, ніж знімок 384², тож за епоху модель
бачить менше «води й кораблів», і епох знадобиться більше, ніж 8 у v1. Остаточне число — за оцінкою часу з пілота.

In [ ]:
import random
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import segmentation_models_pytorch as smp
import torch
from torch.utils.data import DataLoader

PILOT = True

if ON_KAGGLE:
    sys.path.append("/kaggle/input/datasets/yurapolishchuk/airbus-ship-detection-src")
    DATA_DIR = KAGGLE_DATA_DIR
    IMAGES_DIR = DATA_DIR / "train_v2"
    OUT_DIR = Path("/kaggle/working")
    DEVICE = "cuda"
    MAX_TRAIN_BATCHES = 50 if PILOT else None
    MAX_VAL_IMAGES = 1000 if PILOT else None
    EPOCHS = 1 if PILOT else 20
    NUM_WORKERS = 4
    BATCH_SIZE = 16
    VAL_BATCH_SIZE = 8
else:
    ROOT = Path.cwd().parent
    sys.path.append(str(ROOT / "src"))
    DATA_DIR = ROOT / "data"
    IMAGES_DIR = DATA_DIR / "train_micro"
    OUT_DIR = DATA_DIR / "unet_crops_local"
    DEVICE = "cpu"
    MAX_TRAIN_BATCHES = 2
    MAX_VAL_IMAGES = None
    EPOCHS = 1
    NUM_WORKERS = 0
    BATCH_SIZE = 4
    VAL_BATCH_SIZE = 2

from dataset import (CROP_SIZE, IMAGENET_MEAN, IMAGENET_STD, ShipCropDataset, ShipSegmentationDataset,
                     rles_by_image)
from rle import IMAGE_SHAPE
from split import make_split

SEED = 42
LR = 3e-4
SHIP_PROB = 0.75
USE_AMP = DEVICE == "cuda"
OUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("Kaggle:", ON_KAGGLE, "| PILOT:", PILOT, "| device:", DEVICE, "| AMP:", USE_AMP)

### 1.1. Дані

Поділ — та сама `make_split` (seed 42), тож val — ті самі 4 256 знімків, що у v1.
Train — `ShipCropDataset`: з імовірністю `SHIP_PROB` = 0.75 вікно навколо випадково обраного корабля,
інакше — випадкове вікно (складні негативи (hard negatives): вода, берег, хвилі).
Val — `ShipSegmentationDataset(..., size=768)`: знімок без стискання.

In [ ]:
masks = pd.read_csv(DATA_DIR / "train_ship_segmentations_v2.csv")
rles = rles_by_image(masks)
if ON_KAGGLE:
    split = make_split(masks)
else:
    # З 300 знімків train_micro у справжню val потрапляє надто мало — ділимо саму вибірку.
    available = {p.name for p in IMAGES_DIR.iterdir()}
    split = make_split(masks[masks["ImageId"].isin(available)], val_size=0.2)

with_ships = split[split["n_ships"] > 0]
train_df = with_ships[with_ships["split"] == "train"].reset_index(drop=True)
val_df = with_ships[with_ships["split"] == "val"].reset_index(drop=True)
n_val_full = len(val_df)
if MAX_VAL_IMAGES is not None:
    val_df = val_df.sample(n=MAX_VAL_IMAGES, random_state=SEED).reset_index(drop=True)

train_dataset = ShipCropDataset(train_df["ImageId"], rles, IMAGES_DIR, CROP_SIZE, SHIP_PROB)
val_dataset = ShipSegmentationDataset(val_df["ImageId"], rles, IMAGES_DIR, train=False, size=IMAGE_SHAPE[0])
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                          generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(val_dataset, batch_size=VAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"train: {len(train_df)} знімків з кораблями → кропи {CROP_SIZE}×{CROP_SIZE}")
print(f"val:   {len(val_df)} знімків з кораблями, {IMAGE_SHAPE[0]}×{IMAGE_SHAPE[1]}")

### 1.2. Перевірка кропів очима

8 кропів з train **після аугментацій** (як їх бачить модель) з напівпрозорою маскою. Очікуємо: приблизно 3 з 4
з кораблем, корабель не завжди в центрі, кораблі помітно більші, ніж на знімках 384 у `05`,
а маска точно лежить на кораблі.

In [ ]:
def to_display(image_tensor):
    image = image_tensor.numpy().transpose(1, 2, 0) * IMAGENET_STD + IMAGENET_MEAN
    return np.clip(image, 0, 1)


def overlay(image, mask, color=(1.0, 0.0, 0.0)):
    shown = image.copy()
    shown[mask > 0.5] = 0.6 * shown[mask > 0.5] + 0.4 * np.array(color)
    return shown


fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, i in zip(axes.flat, range(8)):
    image, mask = train_dataset[i]
    ax.imshow(overlay(to_display(image), mask[0].numpy()))
    ax.set_title(f"{train_df['ImageId'][i]}, пікселів корабля {int(mask.sum())}")
    ax.axis("off")
plt.tight_layout()
plt.show()

image, mask = val_dataset[0]
print(f"train: знімок {tuple(train_dataset[0][0].shape)} | val: знімок {tuple(image.shape)}, маска {tuple(mask.shape)}")

### 1.3. Модель і loss

Як у `05`: `smp.Unet` з encoder ResNet34 (ImageNet), loss = BCE (binary cross-entropy, бінарна перехресна
ентропія) + Dice. U-Net — повністю згорткова мережа (FCN, fully convolutional network), тож одна й та сама
модель приймає і кроп 256, і знімок 768: обидва розміри діляться на 32.

In [ ]:
model = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", classes=1).to(DEVICE)

bce_loss = torch.nn.BCEWithLogitsLoss()
dice_loss = smp.losses.DiceLoss(mode="binary", from_logits=True)


def loss_fn(logits, masks):
    return bce_loss(logits, masks) + dice_loss(logits, masks)


optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scaler = torch.amp.GradScaler(DEVICE, enabled=USE_AMP)

print(f"ваг у моделі: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} млн")

### 1.4. Метрика: Dice по знімках

Та сама функція, що в `05` (перевірена там на прикладах з відомою відповіддю).

In [ ]:
def dice_per_image(probs, masks, threshold=0.5):
    predicted = (probs > threshold).float()
    intersection = (predicted * masks).sum(dim=(1, 2, 3))
    total = predicted.sum(dim=(1, 2, 3)) + masks.sum(dim=(1, 2, 3))
    return torch.where(total > 0, 2 * intersection / total.clamp(min=1), torch.ones_like(total))

### 1.5. Навчання однієї епохи і оцінка

Код — без змін з `05`: розмір входу береться з батча, тож train іде на `[B, 3, 256, 256]`, а val — на `[B, 3, 768, 768]`.

In [ ]:
def train_one_epoch(model, loader, max_batches=None):
    model.train()
    losses = []
    for i, (images, masks) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        with torch.autocast(device_type=DEVICE, enabled=USE_AMP):
            logits = model(images)
        loss = loss_fn(logits.float(), masks)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.item())
    return np.mean(losses)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    losses, dices = [], []
    for images, masks in loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        with torch.autocast(device_type=DEVICE, enabled=USE_AMP):
            logits = model(images)
        losses.append(loss_fn(logits.float(), masks).item())
        dices.append(dice_per_image(torch.sigmoid(logits.float()), masks).cpu())
    return np.mean(losses), torch.cat(dices).numpy()

### 1.6. Цикл по епохах

Як у `05`, ваги — у `unet_crops_resnet34.pth` (інша назва, щоб не сплутати з v1 в Input наступних ноутбуків).
Train loss рахується на кропах, val loss — на цілих знімках, де води набагато більше: val loss може бути
нижчим за train loss, і це не помилка.

In [ ]:
best_dice = -1
history = []

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_loss = train_one_epoch(model, train_loader, MAX_TRAIN_BATCHES)
    train_seconds = time.time() - start
    val_loss, val_dices = evaluate(model, val_loader)
    val_seconds = time.time() - start - train_seconds
    mean_dice = val_dices.mean()
    minutes = (time.time() - start) / 60
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_dice": mean_dice,
                    "minutes": minutes})
    print(f"епоха {epoch}: train loss {train_loss:.4f} | val loss {val_loss:.4f} | val Dice {mean_dice:.4f} | {minutes:.1f} хв")

    if mean_dice > best_dice:
        best_dice = mean_dice
        torch.save(model.state_dict(), OUT_DIR / "unet_crops_resnet34.pth")
        val_df.assign(dice=val_dices)[["ImageId", "n_ships", "dice"]].to_csv(OUT_DIR / "val_dice.csv", index=False)
        print("    ↑ найкраща епоха — ваги і val_dice.csv збережено")

    if PILOT:
        n_train_seen = min(MAX_TRAIN_BATCHES or len(train_loader), len(train_loader)) * BATCH_SIZE
        train_speed = n_train_seen / train_seconds
        val_speed = len(val_df) / val_seconds
        full_epoch = len(train_df) / train_speed + n_val_full / val_speed
        print(f"    train: {train_speed:.0f} кропів/с, val: {val_speed:.0f} знімків/с")
        print(f"    оцінка повної епохи: {full_epoch / 60:.0f} хв "
              f"({len(train_df)} train-кропів + {n_val_full} val-знімків)")
        print(f"    скільки епох влізе в 11 год (ліміт сесії Kaggle 12 год): {11 * 60 / (full_epoch / 60):.0f}")

pd.DataFrame(history).to_csv(OUT_DIR / "history.csv", index=False)

### 1.7. Приклади передбачень і перевірка збережених ваг

Як у `05`, але знімки 768: правда (червоне), передбачення p > 0.5 (зелене). Dice перезавантаженої моделі
має збігатися зі збереженим.

In [ ]:
reloaded = smp.Unet(encoder_name="resnet34", encoder_weights=None, classes=1)
reloaded.load_state_dict(torch.load(OUT_DIR / "unet_crops_resnet34.pth", map_location=DEVICE))
reloaded = reloaded.to(DEVICE).eval()

_, reloaded_dices = evaluate(reloaded, val_loader)
saved = pd.read_csv(OUT_DIR / "val_dice.csv")
print("макс. різниця Dice після перезавантаження ваг:", np.abs(reloaded_dices - saved["dice"].values).max())

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for col in range(4):
    image, mask = val_dataset[col]
    with torch.no_grad():
        prob = torch.sigmoid(reloaded(image.unsqueeze(0).to(DEVICE))).cpu()[0, 0].numpy()
    shown = to_display(image)
    axes[0, col].imshow(overlay(shown, mask[0].numpy()))
    axes[0, col].set_title(f"правда: {val_df['ImageId'][col]}")
    axes[1, col].imshow(overlay(shown, prob, color=(0.0, 1.0, 0.0)))
    axes[1, col].set_title(f"передбачення, Dice {saved['dice'][col]:.2f}")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT_DIR / "val_examples.png", dpi=80)
plt.show()